# Notebook 04 — Evaluations: measuring how good the AI really is

**Time:** about 45 minutes

**What you will learn**
- Why we **measure** instead of guessing
- How to build a small test set with correct answers
- **Error analysis**: finding *which kind* of input fails
- Improving a prompt and measuring again
- Using a model as a **judge** (and why to be careful)

### The big idea
"It looks good" is not evidence. We need a set of examples with **known correct answers**,
a number (for example, accuracy), and a look at the mistakes.

Our task: read an invoice and find its **payment due date**.

### ▶️ Run the two setup cells below first
They check that Ollama is ready and create `ask()`, the one helper we use to talk to the model.

In [ ]:
# STEP 0 — Is Ollama running? Is the model downloaded?
import ollama

MODEL = "llama3.2"     # local model (bigger one: "qwen2.5:7b")

try:
    installed = [m.model for m in ollama.list().models]
    print("Models on this computer:", installed)
    if MODEL in installed or MODEL + ":latest" in installed:
        print("✅ Ready to go with", MODEL)
    else:
        print(f"⚠️ {MODEL} is missing. Run: ollama pull {MODEL}")
except ConnectionError:
    print("⚠️ Ollama is not running. Start it, then run this again:")
    print("   Arch Linux: sudo systemctl start ollama")
    print("   Windows 11: open Ollama from the Start menu")

In [ ]:
# STEP 1 — A helper: send a message, get the answer back as text
DEFAULT_ROLE = "You are a helpful assistant. Keep answers short."

def ask(prompt, system=DEFAULT_ROLE):
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        options={"temperature": 0},    # 0 = more predictable
    )
    return reply.message.content

print("ask() is ready.")

## Part 1 — The test set
`invoices.csv` has invoice texts written in many styles, and the correct due date for each one.
To keep it quick, we use the first `N` invoices. (Set `N = 18` to use all of them.)

In [ ]:
import pandas as pd

N = 10      # how many invoices to test (18 = all of them)
invoices = pd.read_csv("../datasets/invoices.csv").head(N)
invoices[["invoice_number", "vendor_country", "raw_text",
          "due_date"]]

## Part 2 — Ask the model for a due date, as JSON

We want a clean answer like `{"due_date": "2025-04-03"}`, not a long sentence.
Ollama can force the reply to follow a **schema**. We describe the schema with a small Pydantic class.

In [ ]:
from pydantic import BaseModel

class DueDate(BaseModel):
    due_date: str          # in the format YYYY-MM-DD

def extract_due_date(invoice_text, instructions):
    prompt = instructions + "\n\nINVOICE: " + invoice_text
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        format=DueDate.model_json_schema(),   # reply must be JSON
        options={"temperature": 0},
    )
    try:
        answer = DueDate.model_validate_json(reply.message.content)
        return answer.due_date
    except Exception:
        return "invalid answer"     # a broken reply counts as wrong

In [ ]:
PROMPT_V1 = ("What is the payment due date of this invoice? "
             "Answer in the format YYYY-MM-DD.")

first_invoice = invoices.loc[0, "raw_text"]
print(first_invoice)
print("→", extract_due_date(first_invoice, PROMPT_V1))

## Part 3 — Run the whole test set and measure accuracy
This makes `N` model calls, so it takes a little while.

In [ ]:
COLUMNS = ["invoice_number", "vendor_country", "format_type",
           "raw_text", "due_date"]

def evaluate(prompt):
    results = invoices[COLUMNS].copy()
    predictions = []
    for text in results["raw_text"]:
        predictions.append(extract_due_date(text, prompt))
    results["predicted"] = predictions
    results["correct"] = results["predicted"] == results["due_date"]
    return results

results_v1 = evaluate(PROMPT_V1)
right = results_v1["correct"].sum()
print(f"Prompt v1: {right} of {len(results_v1)} correct")
results_v1[["invoice_number", "due_date", "predicted", "correct"]]

## Part 4 — Error analysis: which kind of invoice fails?
Each invoice has a `format_type` (for example `us_slash` = 04/09/2025, `ambiguous_dd_mm` = British day/month order).
Grouping the mistakes by type tells us **what to fix**.

In [ ]:
mistakes = results_v1[results_v1["correct"] == False]

print("Wrong answers by invoice type:")
print(mistakes["format_type"].value_counts())
mistakes[["vendor_country", "format_type", "raw_text",
          "due_date", "predicted"]]

👉 **Discuss:** look at the wrong rows. Is the model confusing day and month? Missing "due on receipt"?

## Part 5 — Improve the prompt, then measure again
We add the rules a careful human would use, and we tell the model the vendor's **country**
(in the US dates are month/day; in most other countries day/month).

In [ ]:
PROMPT_V2 = """Find the PAYMENT DUE DATE (not the issue date).
Rules:
- Answer in the format YYYY-MM-DD.
- Vendor country US: 04/09/2025 means MONTH/DAY/YEAR.
- Any other country: 04/09/2025 means DAY/MONTH/YEAR.
- "Due on receipt" means the due date is the issue date.
- "Net 30" means 30 days after the issue date (Net 45 = 45).
- "Amount due" is money, not a date."""

def evaluate_v2():
    results = invoices[COLUMNS].copy()
    predictions = []
    for country, text in zip(results["vendor_country"],
                             results["raw_text"]):
        invoice = f"Vendor country: {country}. {text}"
        predictions.append(extract_due_date(invoice, PROMPT_V2))
    results["predicted"] = predictions
    results["correct"] = results["predicted"] == results["due_date"]
    return results

In [ ]:
results_v2 = evaluate_v2()

v1 = results_v1["correct"].sum()
v2 = results_v2["correct"].sum()
print(f"Prompt v1: {v1} of {N} correct")
print(f"Prompt v2: {v2} of {N} correct")

⚠️ With only `N` examples, a difference of one or two answers can be luck. Real evaluations use more examples,
and you should never tune your prompt on the **same** examples you report as your final score.

## Part 6 — A model as a judge (for answers with no single correct value)

Some answers cannot be checked with `==` (an essay, an explanation). Then we can ask a model to **grade** them
with a clear rubric. But a judge model can be wrong, so we compare it with our own (human) judgement.

In [ ]:
class Grade(BaseModel):
    score: int        # 1 (poor) to 5 (excellent)
    reason: str

RUBRIC = ("Grade the answer from 1 to 5. 5 = correct, complete "
          "and clear. 1 = wrong or missing the point. "
          "Give a one-sentence reason.")

def judge(question, answer):
    prompt = f"{RUBRIC}\n\nQUESTION: {question}\n\nANSWER: {answer}"
    message = {"role": "user", "content": prompt}
    reply = ollama.chat(model=MODEL, messages=[message],
                        format=Grade.model_json_schema(),
                        options={"temperature": 0})
    try:
        return Grade.model_validate_json(reply.message.content)
    except Exception:
        return Grade(score=0, reason="Invalid judge answer.")

In [ ]:
question = "Why should an AI agent have a limit on its steps?"
answers = {
    "good (human: 5)": "So it cannot loop forever: a step limit "
                       "stops runaway time and cost.",
    "poor (human: 1)": "Because agents are made of steps.",
}

for label, answer in answers.items():
    grade = judge(question, answer)
    print(f"{label}: judge score {grade.score} — {grade.reason}")

## Part 7 — Two kinds of evaluation

| | Checked by **code** | Checked by a **judge model** |
|---|---|---|
| Example | due date == correct date | "Is this explanation clear?" |
| Reliable? | Yes, same result every time | Can be wrong; check it against people |
| Use when | There is one correct answer | Quality is a matter of judgement |

## ✍️ Try it yourself
1. Set `N = 18` and run Parts 1–5 again. Does the improvement still hold?
2. Write a third answer for the judge that is half-correct. What score does it get? Do you agree?

## ✅ Summary
- Build a small test set with correct answers, measure a number, then **read the mistakes**.
- Group mistakes by type to decide what to fix; then measure again.
- Judge models are useful for open-ended answers, but must be checked against human judgement.

**Next:** Notebook 05 — **planning**: the model writes a plan, our code checks and runs it.